# AnimalCLEF2026 SAM3 Masked ReID Fusion v20260423

This is the stronger segmentation-first notebook:

- SAM3 segments the animal and erases the background.
- MegaDescriptor-L-384 + MiewID-msv3 extract embeddings from the cleaned animal crops.
- Open-set validation tunes species-specific graph thresholds.
- Train identities are used only as safe anchors, not blind nearest-neighbor labels.
- Optional SIFT verification adjusts final test-test graph edges near the threshold.

Run on an L4/T4/P100 GPU with Internet enabled. If SAM3 is gated, set `HF_TOKEN` as a Kaggle Secret or environment variable.

In [ ]:
import importlib.util, subprocess, sys

needed = {
    'timm': 'timm>=0.9.16',
    'transformers': 'transformers>=4.40.0',
    'cv2': 'opencv-python-headless',
    'sklearn': 'scikit-learn',
    'tqdm': 'tqdm',
    'PIL': 'pillow',
    'huggingface_hub': 'huggingface_hub',
}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('[setup] installing', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

try:
    from transformers import Sam3Processor, Sam3Model
    print('[setup] SAM3 available')
except Exception:
    print('[setup] upgrading transformers for SAM3')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers'])
    try:
        from transformers import Sam3Processor, Sam3Model
    except Exception:
        print('[setup] installing transformers from GitHub main')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'git+https://github.com/huggingface/transformers'])
        from transformers import Sam3Processor, Sam3Model

print('[setup] ready')


In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import gc, json, math, os, random, time

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from sklearn.metrics import adjusted_rand_score

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from huggingface_hub import login
from transformers import Sam3Processor, Sam3Model

ImageFile.LOAD_TRUNCATED_IMAGES = True
VERSION = 'sam3_masked_reid_fusion_v20260423'
SEED = 20260423
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT_DIR = WORK_DIR / 'animalclef_sam3_masked_reid'
CROP_DIR = OUT_DIR / 'masked_crops'
CACHE_DIR = OUT_DIR / 'cache'
REPORT_DIR = OUT_DIR / 'reports'
SUB_DIR = OUT_DIR / 'submissions'
for d in [OUT_DIR, CROP_DIR, CACHE_DIR, REPORT_DIR, SUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0))

token = os.environ.get('HF_TOKEN')
if token is None:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        token = None
if token:
    login(token=token, add_to_git_credential=False)
    print('[hf] logged in')
else:
    print('[hf] no HF_TOKEN found')

def find_competition_root():
    candidates = [Path('/kaggle/input/animal-clef-2026'), Path('/kaggle/input/competitions/animal-clef-2026'), Path('/content/animal-clef-2026'), Path.cwd() / 'animal-clef-2026']
    for root in candidates:
        if (root / 'metadata.csv').exists() and (root / 'sample_submission.csv').exists():
            return root
    for base in [Path('/kaggle/input'), Path('/content'), Path.cwd()]:
        if base.exists():
            for p in base.rglob('metadata.csv'):
                if (p.parent / 'sample_submission.csv').exists():
                    return p.parent
    raise FileNotFoundError('AnimalCLEF2026 data root not found')

DATA_ROOT = find_competition_root()
meta = pd.read_csv(DATA_ROOT / 'metadata.csv').reset_index(drop=True)
sample = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
meta['row_idx'] = np.arange(len(meta))
meta['orientation'] = meta.get('orientation', '').fillna('').astype(str)
meta['date_dt'] = pd.to_datetime(meta.get('date'), errors='coerce')
meta['abs_path'] = meta['path'].map(lambda p: str((DATA_ROOT / str(p)).resolve()) if not Path(str(p)).is_absolute() else str(p))
print('DATA_ROOT:', DATA_ROOT)
print(meta.groupby(['dataset', 'split']).size())


In [ ]:
# Main controls.
SEGMENT_TRAIN_AND_TEST = True
SAM3_MAX_SIDE = 896
SAM3_BATCH_SIZE = 8
BACKGROUND_COLOR = (238, 238, 232)
RUN_SIFT_EDGE_VERIFY = True

PROMPT = {
    'LynxID2025': 'lynx animal',
    'SalamanderID2025': 'salamander animal',
    'SeaTurtleID2022': 'sea turtle head animal',
    'TexasHornedLizards': 'horned lizard animal',
}
SAM3_THRESHOLD = {'LynxID2025': 0.35, 'SalamanderID2025': 0.30, 'SeaTurtleID2022': 0.35, 'TexasHornedLizards': 0.30}
SAM3_MASK_THRESHOLD = 0.50

BATCH_SIZE_MEGA = 48
BATCH_SIZE_MIEW = 64
NUM_WORKERS = 2

TOPK_GRID = [10, 20, 40]
EDGE_GRID = np.round(np.linspace(0.44, 0.82, 14), 3).tolist()
KNOWN_GRID = [0.78, 0.84, 0.90, 0.96]
MARGIN_THR = 0.04

FINAL_TOPK_FALLBACK = {'LynxID2025': 25, 'SalamanderID2025': 25, 'SeaTurtleID2022': 25, 'TexasHornedLizards': 30}
FINAL_EDGE_FALLBACK = {'LynxID2025': 0.62, 'SalamanderID2025': 0.64, 'SeaTurtleID2022': 0.62, 'TexasHornedLizards': 0.63}
FINAL_KNOWN_FALLBACK = {'LynxID2025': 0.90, 'SalamanderID2025': 0.96, 'SeaTurtleID2022': 0.90, 'TexasHornedLizards': 1.10}
MAX_COMPONENT_SIZE = {'LynxID2025': 70, 'SalamanderID2025': 25, 'SeaTurtleID2022': 35, 'TexasHornedLizards': 30}
MAX_ANCHOR_BUCKET = {'LynxID2025': 24, 'SalamanderID2025': 12, 'SeaTurtleID2022': 18, 'TexasHornedLizards': 0}

SIFT_PAIR_CAP = {'LynxID2025': 5000, 'SalamanderID2025': 4500, 'SeaTurtleID2022': 3200, 'TexasHornedLizards': 2200}
SIFT_MIN_GOOD = {'LynxID2025': 24, 'SalamanderID2025': 16, 'SeaTurtleID2022': 18, 'TexasHornedLizards': 16}
print('version:', VERSION)


In [ ]:
print('[sam3] loading model')
sam_model = Sam3Model.from_pretrained('facebook/sam3').to(DEVICE).eval()
sam_processor = Sam3Processor.from_pretrained('facebook/sam3')
print('[sam3] ready')

def load_resized_rgb(path, max_side=SAM3_MAX_SIDE):
    img = Image.open(path).convert('RGB')
    w, h = img.size
    scale = min(1.0, max_side / max(w, h))
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return img

def choose_mask(masks, scores):
    if masks is None or len(masks) == 0:
        return None
    arr = masks.detach().cpu().numpy().astype(bool)
    scores_np = scores.detach().cpu().numpy() if scores is not None else np.ones(len(arr), dtype='float32')
    h, w = arr.shape[-2:]
    area_total = float(h * w)
    best, best_score = None, -1
    for mask, score in zip(arr, scores_np):
        area = float(mask.sum())
        frac = area / area_total
        if frac < 0.004 or frac > 0.92:
            continue
        ys, xs = np.where(mask)
        if len(xs) == 0:
            continue
        cx = (xs.mean() / max(w - 1, 1)) - 0.5
        cy = (ys.mean() / max(h - 1, 1)) - 0.5
        centrality = 1.0 - min(1.0, math.sqrt(cx * cx + cy * cy) / 0.70)
        rank = float(score) * 0.68 + min(frac / 0.33, 1.0) * 0.18 + centrality * 0.14
        if rank > best_score:
            best_score = rank
            best = mask
    return best

def save_masked_crop(path, species, image, mask):
    path = Path(path)
    out_dir = CROP_DIR / species
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / (path.stem + '_masked.jpg')
    if out_path.exists():
        return str(out_path), True
    img = np.array(image)
    if mask is None:
        crop, ok = img, False
    else:
        ys, xs = np.where(mask)
        if len(xs) == 0:
            crop, ok = img, False
        else:
            h, w = mask.shape
            margin = int(0.10 * max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1)) + 10
            x0, x1 = max(0, xs.min() - margin), min(w, xs.max() + margin + 1)
            y0, y1 = max(0, ys.min() - margin), min(h, ys.max() + margin + 1)
            bg = np.zeros_like(img)
            bg[:, :] = np.array(BACKGROUND_COLOR, dtype=np.uint8)
            masked = np.where(mask[..., None], img, bg)
            crop = masked[y0:y1, x0:x1]
            ok = True
    if min(crop.shape[:2]) < 64:
        crop, ok = img, False
    cv2.imwrite(str(out_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
    return str(out_path), ok

def segment_batch(paths, species, batch_size=SAM3_BATCH_SIZE):
    paths = [str(p) for p in paths]
    outputs = [None] * len(paths)
    pending_slots, pending_paths, pending_images = [], [], []
    for slot, path in enumerate(paths):
        p = Path(path)
        out_path = CROP_DIR / species / (p.stem + '_masked.jpg')
        if out_path.exists():
            outputs[slot] = (str(out_path), True)
        else:
            pending_slots.append(slot)
            pending_paths.append(p)
            pending_images.append(load_resized_rgb(p))
    prompt = PROMPT[species]
    for start in tqdm(range(0, len(pending_paths), batch_size), desc=f'SAM3 mask {species}'):
        batch_slots = pending_slots[start:start + batch_size]
        batch_paths = pending_paths[start:start + batch_size]
        batch_images = pending_images[start:start + batch_size]
        masks = [None] * len(batch_images)
        try:
            inputs = sam_processor(images=batch_images, text=[prompt] * len(batch_images), return_tensors='pt').to(DEVICE)
            with torch.inference_mode():
                if DEVICE == 'cuda':
                    with torch.autocast(device_type='cuda', dtype=torch.float16):
                        out = sam_model(**inputs)
                else:
                    out = sam_model(**inputs)
            results = sam_processor.post_process_instance_segmentation(out, threshold=SAM3_THRESHOLD[species], mask_threshold=SAM3_MASK_THRESHOLD, target_sizes=inputs.get('original_sizes').tolist())
            for k, result in enumerate(results):
                masks[k] = choose_mask(result.get('masks'), result.get('scores'))
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and DEVICE == 'cuda' and batch_size > 1:
                torch.cuda.empty_cache()
                print(f'[sam3] OOM at batch_size={batch_size}; retrying one by one')
                retry = segment_batch([str(p) for p in batch_paths], species, batch_size=1)
                for slot, result in zip(batch_slots, retry):
                    outputs[slot] = result
                continue
            print('[sam3] batch failed:', repr(e))
        except Exception as e:
            print('[sam3] batch failed:', repr(e))
        for slot, p, img, mask in zip(batch_slots, batch_paths, batch_images, masks):
            outputs[slot] = save_masked_crop(p, species, img, mask)
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    return outputs

crop_path_by_row = {}
mask_ok_by_row = {}
target_meta = meta if SEGMENT_TRAIN_AND_TEST else meta[meta['split'].eq('test')]
for species, g in target_meta.groupby('dataset'):
    results = segment_batch(g['abs_path'].tolist(), species)
    ok_count = sum(ok for _, ok in results)
    print(f'[sam3] {species}: masks ok {ok_count}/{len(results)}')
    for row_idx, (crop, ok) in zip(g['row_idx'].tolist(), results):
        crop_path_by_row[int(row_idx)] = crop
        mask_ok_by_row[int(row_idx)] = bool(ok)

meta['crop_path'] = meta.apply(lambda r: crop_path_by_row.get(int(r['row_idx']), r['abs_path']), axis=1)
meta['mask_ok'] = meta['row_idx'].map(lambda x: mask_ok_by_row.get(int(x), False))
pd.DataFrame({'row_idx': list(crop_path_by_row.keys()), 'crop_path': list(crop_path_by_row.values())}).to_csv(REPORT_DIR / f'crop_manifest_{VERSION}.csv', index=False)
del sam_model
gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()


In [ ]:
class PathImageDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (512, 512), BACKGROUND_COLOR)
        return self.transform(img)

def l2_normalize(x):
    x = np.asarray(x, dtype=np.float32)
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def output_to_tensor(out):
    if isinstance(out, torch.Tensor):
        x = out
    elif isinstance(out, dict):
        for key in ['embedding', 'embeddings', 'features', 'pooler_output', 'last_hidden_state']:
            if key in out and out[key] is not None:
                x = out[key]
                break
        else:
            raise TypeError(f'Unsupported output keys: {list(out.keys())}')
    elif hasattr(out, 'pooler_output') and out.pooler_output is not None:
        x = out.pooler_output
    elif hasattr(out, 'last_hidden_state'):
        x = out.last_hidden_state.mean(dim=1)
    elif isinstance(out, (tuple, list)):
        x = out[0]
    else:
        raise TypeError(type(out))
    if x.ndim == 4:
        x = torch.nn.functional.adaptive_avg_pool2d(x, 1).flatten(1)
    if x.ndim == 3:
        x = x.mean(dim=1)
    return x

def extract_features(model, paths, transform, cache_path, batch_size):
    cache_path = Path(cache_path)
    if cache_path.exists():
        arr = np.load(cache_path)
        if arr.shape[0] == len(paths):
            print('[cache] loaded', cache_path, arr.shape)
            return l2_normalize(arr)
    ds = PathImageDataset(paths, transform)
    bs = int(batch_size)
    while bs >= 1:
        try:
            loader = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'))
            feats = []
            model.eval()
            with torch.inference_mode():
                for batch in tqdm(loader, desc=f'extract bs={bs}'):
                    batch = batch.to(DEVICE, non_blocking=True)
                    if DEVICE == 'cuda':
                        with torch.autocast(device_type='cuda', dtype=torch.float16):
                            out = model(batch)
                    else:
                        out = model(batch)
                    feats.append(output_to_tensor(out).float().cpu().numpy())
            arr = l2_normalize(np.concatenate(feats, axis=0))
            np.save(cache_path, arr.astype(np.float16))
            print('[cache] wrote', cache_path, arr.shape)
            return arr
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower() or bs == 1:
                raise
            print('[oom] feature batch', bs, '->', bs // 2)
            bs = max(1, bs // 2)
            if DEVICE == 'cuda': torch.cuda.empty_cache()

def load_mega():
    import timm
    model = timm.create_model('hf-hub:BVRA/MegaDescriptor-L-384', pretrained=True, num_classes=0)
    return model.to(DEVICE)

def load_miew():
    from transformers import AutoModel
    from transformers.modeling_utils import PreTrainedModel
    if not hasattr(PreTrainedModel, 'all_tied_weights_keys'):
        PreTrainedModel.all_tied_weights_keys = {}
    model = AutoModel.from_pretrained('conservationxlabs/miewid-msv3', trust_remote_code=True)
    if not hasattr(model, 'all_tied_weights_keys'):
        model.all_tied_weights_keys = {}
    return model.to(DEVICE)

crop_paths = meta['crop_path'].tolist()
mega_transform = T.Compose([T.Resize((384, 384)), T.ToTensor(), T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
miew_transform = T.Compose([T.Resize((440, 440)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

mega_model = load_mega()
mega = extract_features(mega_model, crop_paths, mega_transform, CACHE_DIR / 'mega_masked_all.npy', BATCH_SIZE_MEGA)
del mega_model
gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

miew_model = load_miew()
miew = extract_features(miew_model, crop_paths, miew_transform, CACHE_DIR / 'miew_masked_all.npy', BATCH_SIZE_MIEW)
del miew_model
gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

features = l2_normalize(np.concatenate([0.66 * mega, 0.34 * miew], axis=1))
print('features', features.shape)


In [ ]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n)); self.sz = [1] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b, max_size=None):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return True
        if max_size is not None and self.sz[ra] + self.sz[rb] > max_size:
            return False
        if self.sz[ra] < self.sz[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra; self.sz[ra] += self.sz[rb]
        return True

def side_bucket(x):
    s = str(x).lower()
    if 'left' in s: return 'left'
    if 'right' in s: return 'right'
    return None

def orient_penalty(sim, q_meta, g_meta, species):
    if species not in {'LynxID2025', 'SeaTurtleID2022'}:
        return sim
    q_side = [side_bucket(x) for x in q_meta['orientation'].tolist()]
    g_side = np.array([side_bucket(x) for x in g_meta['orientation'].tolist()], dtype=object)
    pen = 0.065 if species == 'LynxID2025' else 0.045
    for i, s in enumerate(q_side):
        if s is None: continue
        mask = np.array([(t is not None and t != s) for t in g_side])
        if mask.any(): sim[i, mask] -= pen
    return sim

def build_edges(query_idx, species, topk):
    q_meta = meta.loc[query_idx].reset_index(drop=True)
    q_feat = features[query_idx]
    sim = q_feat @ q_feat.T
    sim = orient_penalty(sim, q_meta, q_meta, species)
    np.fill_diagonal(sim, -9.0)
    n = len(query_idx)
    k = min(int(topk), max(1, n - 1))
    nbr = np.argpartition(-sim, kth=np.arange(k), axis=1)[:, :k]
    nbr_sets = [set(row.tolist()) for row in nbr]
    edges = []
    for i in range(n):
        for j in nbr[i]:
            j = int(j)
            if j <= i: continue
            if i in nbr_sets[j]:
                edges.append((float(sim[i, j]), i, j))
    edges.sort(reverse=True)
    return edges, sim, q_meta

def predict_graph(query_idx, gallery_idx, species, topk, edge_thr, known_thr, margin_thr=MARGIN_THR, edges_override=None):
    edges, sim, q_meta = build_edges(query_idx, species, topk)
    if edges_override is not None:
        edges = edges_override
    n = len(query_idx)
    dsu = DSU(n)
    for score, i, j in edges:
        if score < edge_thr: break
        dsu.union(i, j, max_size=MAX_COMPONENT_SIZE[species])
    if len(gallery_idx) > 0 and known_thr < 1.0 and MAX_ANCHOR_BUCKET[species] > 0:
        g_meta = meta.loc[gallery_idx].reset_index(drop=True)
        qg = features[query_idx] @ features[gallery_idx].T
        qg = orient_penalty(qg, q_meta, g_meta, species)
        order = np.argpartition(-qg, kth=np.arange(min(2, qg.shape[1])), axis=1)[:, :min(2, qg.shape[1])]
        glabs = g_meta['identity'].astype(str).to_numpy()
        buckets = defaultdict(list)
        for i in range(n):
            cand = order[i]
            cand = cand[np.argsort(-qg[i, cand])]
            top_score = float(qg[i, cand[0]])
            second = float(qg[i, cand[1]]) if len(cand) > 1 else -1.0
            if top_score >= known_thr and (top_score - second) >= margin_thr:
                buckets[glabs[int(cand[0])]].append(i)
        pair_thr = max(edge_thr - 0.035, 0.50)
        for lab, inds in buckets.items():
            if len(inds) > MAX_ANCHOR_BUCKET[species]:
                continue
            for a in range(len(inds)):
                i = inds[a]
                for j in inds[a + 1:]:
                    if sim[i, j] >= pair_thr:
                        dsu.union(i, j, max_size=MAX_COMPONENT_SIZE[species])
    roots, pred = {}, []
    for i in range(n):
        r = dsu.find(i)
        if r not in roots:
            roots[r] = len(roots)
        pred.append(roots[r])
    return np.array(pred), edges

def open_split(species, fold=0):
    df = meta[(meta['dataset'] == species) & (meta['split'] == 'train')].copy()
    rng = np.random.default_rng(SEED + fold * 101 + sum(map(ord, species)))
    ids = np.array(sorted(df['identity'].dropna().unique()), dtype=object).copy()
    rng.shuffle(ids)
    novel = set(ids[:max(1, int(0.22 * len(ids)))])
    gallery, query = [], []
    for ident, g in df.groupby('identity'):
        rows = g['row_idx'].to_numpy().copy(); rng.shuffle(rows)
        if ident in novel:
            query.extend(rows.tolist())
        elif len(rows) >= 2:
            query.append(int(rows[0])); gallery.extend(rows[1:].tolist())
        else:
            gallery.extend(rows.tolist())
    return np.array(gallery, dtype=int), np.array(query, dtype=int)

tuned = {}
for species in ['LynxID2025', 'SalamanderID2025', 'SeaTurtleID2022']:
    print('\n[tune]', species)
    best = (-9.0, None)
    for topk in TOPK_GRID:
        for edge_thr in EDGE_GRID:
            for known_thr in KNOWN_GRID:
                aris = []
                for fold in range(2):
                    gallery_idx, query_idx = open_split(species, fold)
                    pred, _ = predict_graph(query_idx, gallery_idx, species, topk, float(edge_thr), float(known_thr))
                    true = meta.loc[query_idx, 'identity'].astype(str).to_numpy()
                    aris.append(adjusted_rand_score(true, pred))
                score = float(np.mean(aris))
                if score > best[0]:
                    best = (score, {'topk': int(topk), 'edge_thr': float(edge_thr), 'known_thr': float(known_thr), 'fold_ari': [float(x) for x in aris]})
    tuned[species] = best[1] | {'local_ari': best[0]}
    print(tuned[species])
tuned['TexasHornedLizards'] = {'topk': FINAL_TOPK_FALLBACK['TexasHornedLizards'], 'edge_thr': FINAL_EDGE_FALLBACK['TexasHornedLizards'], 'known_thr': 1.10, 'local_ari': 'N/A'}
(REPORT_DIR / f'thresholds_{VERSION}.json').write_text(json.dumps(tuned, indent=2))


In [ ]:
sift = cv2.SIFT_create(nfeatures=1600, contrastThreshold=0.018, edgeThreshold=12, sigma=1.2)
bf = cv2.BFMatcher(cv2.NORM_L2)
sift_cache = {}

def rootsift(desc):
    if desc is None or len(desc) == 0: return None
    desc = desc.astype('float32')
    desc /= desc.sum(axis=1, keepdims=True) + 1e-12
    desc = np.sqrt(desc)
    desc /= np.linalg.norm(desc, axis=1, keepdims=True) + 1e-12
    return desc

def get_sift(path):
    if path in sift_cache: return sift_cache[path]
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        result = ([], None)
    else:
        img = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)
        kp, desc = sift.detectAndCompute(img, None)
        result = (kp, rootsift(desc))
    sift_cache[path] = result
    return result

def sift_score(path_a, path_b):
    kp1, d1 = get_sift(path_a); kp2, d2 = get_sift(path_b)
    if d1 is None or d2 is None or len(d1) < 8 or len(d2) < 8:
        return 0.0, 0, 0.0
    try:
        matches = bf.knnMatch(d1, d2, k=2)
    except Exception:
        return 0.0, 0, 0.0
    good = []
    for pair in matches:
        if len(pair) < 2: continue
        m, n = pair
        if m.distance < 0.76 * n.distance:
            good.append(m)
    inlier = 0.0
    if len(good) >= 8:
        pts1 = np.float32([kp1[m.queryIdx].pt for m in good])
        pts2 = np.float32([kp2[m.trainIdx].pt for m in good])
        try:
            _, mask = cv2.estimateAffinePartial2D(pts1, pts2, method=cv2.RANSAC, ransacReprojThreshold=5.0, maxIters=1000)
            if mask is not None: inlier = float(mask.mean())
        except Exception:
            inlier = 0.0
    score = 0.72 * min(1.0, len(good) / 80.0) + 0.28 * inlier
    return float(score), int(len(good)), float(inlier)

def sift_rerank_edges(species, query_idx, edges, edge_thr):
    if not RUN_SIFT_EDGE_VERIFY:
        return edges, {'enabled': False, 'pairs': 0}
    q_meta = meta.loc[query_idx].reset_index(drop=True)
    cap = SIFT_PAIR_CAP[species]
    candidates = [e for e in edges if e[0] >= edge_thr - 0.07]
    candidates.sort(key=lambda e: abs(e[0] - edge_thr))
    candidates = candidates[:cap]
    score_map = {}
    for score, i, j in tqdm(candidates, desc=f'SIFT verify {species}'):
        ss, good, inlier = sift_score(q_meta.loc[i, 'crop_path'], q_meta.loc[j, 'crop_path'])
        new_score = 0.84 * score + 0.16 * ss
        if good >= SIFT_MIN_GOOD[species] and ss > 0.30:
            new_score = max(new_score, edge_thr + 0.01)
        if good < 6 and score < edge_thr + 0.02:
            new_score = min(new_score, edge_thr - 0.01)
        score_map[(i, j)] = float(new_score)
    out = [(score_map.get((i, j), score), i, j) for score, i, j in edges]
    out.sort(reverse=True)
    return out, {'enabled': True, 'pairs': len(score_map)}

parts, run_reports = [], []
for species in ['LynxID2025', 'SalamanderID2025', 'SeaTurtleID2022', 'TexasHornedLizards']:
    rows = meta[(meta['dataset'] == species) & (meta['split'] == 'test')].copy().reset_index(drop=True)
    query_idx = rows['row_idx'].to_numpy()
    gallery_idx = meta[(meta['dataset'] == species) & (meta['split'] == 'train')]['row_idx'].to_numpy()
    cfg = tuned.get(species, {})
    topk = int(cfg.get('topk', FINAL_TOPK_FALLBACK[species]))
    edge_thr = float(cfg.get('edge_thr', FINAL_EDGE_FALLBACK[species]))
    known_thr = float(cfg.get('known_thr', FINAL_KNOWN_FALLBACK[species]))
    print('\n[predict]', species, 'topk', topk, 'edge_thr', edge_thr, 'known_thr', known_thr)
    _, edges = predict_graph(query_idx, gallery_idx, species, topk, edge_thr, known_thr)
    edges2, sift_info = sift_rerank_edges(species, query_idx, edges, edge_thr)
    pred, _ = predict_graph(query_idx, gallery_idx, species, topk, edge_thr, known_thr, edges_override=edges2)
    labels = [f'cluster_{species}_{x}' for x in pred]
    part = pd.DataFrame({'image_id': rows['image_id'].to_numpy(), 'cluster': labels})
    vc = part['cluster'].value_counts()
    report = {'species': species, 'n_images': int(len(part)), 'n_clusters': int(part['cluster'].nunique()), 'max_cluster_size': int(vc.max()), 'singletons': int((vc == 1).sum()), 'sift': sift_info, 'thresholds': cfg}
    print('[report]', report)
    parts.append(part); run_reports.append(report)

sub = pd.concat(parts, ignore_index=True)
sub = sample[['image_id']].merge(sub, on='image_id', how='left')
assert len(sub) == len(sample)
assert sub['cluster'].notna().all()
versioned = SUB_DIR / f'submission_{VERSION}.csv'
sub.to_csv(versioned, index=False)
sub.to_csv(WORK_DIR / 'submission.csv', index=False)
summary = {'version': VERSION, 'thresholds': tuned, 'reports': run_reports, 'submission': str(versioned)}
(REPORT_DIR / f'run_summary_{VERSION}.json').write_text(json.dumps(summary, indent=2))
print('wrote', versioned)
print('wrote', WORK_DIR / 'submission.csv')
print(json.dumps(summary, indent=2)[:5000])
